# Prosody Pitch Feature Extraction (IEMOCAP)

This notebook extracts F0 pitch contours with voicing masks.
Each utterance becomes one training row for downstream SER models.

In [ ]:
from pathlib import Path
import os
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

import librosa
import numpy as np
import pandas as pd


In [ ]:
# Configuration
REPO_ROOT = Path.cwd().parents[1]  # repo root (notebook is under feature_extraction/)
CSV_PATH = REPO_ROOT / "datasets" / "IEMOCAP" / "iemocap_full_dataset.csv"
AUDIO_ROOT = REPO_ROOT / "datasets" / "IEMOCAP"
OUT_DIR = REPO_ROOT / "extracted_features" / "prosody_pitch"
OUT_FILE = "prosody_pitch_features.csv"

# Audio + feature params
TARGET_SR = 16_000
FMIN = 50.0
FMAX = 400.0
FRAME_LENGTH = 4096
HOP_LENGTH = 512
RMS_DB_THRESHOLD = -55.0
RMS_DB_PERCENTILE = 15.0
VOICING_PROB_THRESHOLD = 0.4
SILENCE_TOP_DB = 50.0
MAX_GAP_FRAMES = 4
MIN_VOICED_FRAMES = 3
MIN_VOICING_PROB_MEAN = 0.1

EXCLUDED_EMOTIONS = {"sur", "fea", "oth", "dis"}

OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / OUT_FILE
OUT_PATH


WindowsPath('f:/Speech-Emotion-Recognition/extracted_features/prosody_pitch/prosody_pitch_features.csv')

In [ ]:
def load_audio(path: Path) -> tuple[np.ndarray, int]:
    # Load audio and resample to TARGET_SR so features are comparable
    audio, sr = librosa.load(path, sr=TARGET_SR, mono=True)
    return audio, sr


def _fill_short_gaps(mask: np.ndarray, max_gap_frames: int) -> np.ndarray:
    # Fill small False gaps in a boolean mask
    if max_gap_frames <= 0:
        return mask

    filled = mask.copy()
    gap_start = None
    for idx, is_true in enumerate(mask):
        if not is_true and gap_start is None:
            gap_start = idx
        if is_true and gap_start is not None:
            gap_len = idx - gap_start
            if gap_len <= max_gap_frames:
                filled[gap_start:idx] = True
            gap_start = None
    return filled


def _drop_short_runs(mask: np.ndarray, min_run: int) -> np.ndarray:
    # Drop short True runs shorter than min_run frames
    if min_run <= 1:
        return mask

    cleaned = mask.copy()
    run_start = None
    for idx, is_true in enumerate(mask):
        if is_true and run_start is None:
            run_start = idx
        if not is_true and run_start is not None:
            run_len = idx - run_start
            if run_len < min_run:
                cleaned[run_start:idx] = False
            run_start = None
    if run_start is not None:
        run_len = len(mask) - run_start
        if run_len < min_run:
            cleaned[run_start:] = False
    return cleaned


def compute_pitch(audio: np.ndarray, sr: int) -> tuple[np.ndarray, np.ndarray, np.ndarray | None]:
    # Compute F0 with voicing logic
    f0, _, voiced_prob = librosa.pyin(
        audio,
        fmin=FMIN,
        fmax=FMAX,
        sr=sr,
        frame_length=FRAME_LENGTH,
        hop_length=HOP_LENGTH,
    )
    if np.nanmean(voiced_prob) < MIN_VOICING_PROB_MEAN:
        f0 = librosa.yin(
            audio,
            fmin=FMIN,
            fmax=FMAX,
            sr=sr,
            frame_length=FRAME_LENGTH,
            hop_length=HOP_LENGTH,
        )
        voiced_prob = None

    rms = librosa.feature.rms(
        y=audio,
        frame_length=FRAME_LENGTH,
        hop_length=HOP_LENGTH,
    )[0]
    rms_db = librosa.amplitude_to_db(rms, ref=np.max)
    if RMS_DB_PERCENTILE is None:
        rms_db_effective = RMS_DB_THRESHOLD
    else:
        rms_db_effective = max(
            RMS_DB_THRESHOLD,
            float(np.percentile(rms_db, RMS_DB_PERCENTILE)),
        )

    if voiced_prob is None:
        voiced_mask = rms_db >= rms_db_effective
    else:
        voiced_mask = (rms_db >= rms_db_effective) | (
            voiced_prob >= VOICING_PROB_THRESHOLD
        )

    nonsilent_intervals = librosa.effects.split(
        audio,
        top_db=SILENCE_TOP_DB,
        frame_length=FRAME_LENGTH,
        hop_length=HOP_LENGTH,
    )
    nonsilent_mask = np.zeros_like(voiced_mask, dtype=bool)
    for start, end in nonsilent_intervals:
        start_frame = start // HOP_LENGTH
        end_frame = int(np.ceil(end / HOP_LENGTH))
        nonsilent_mask[start_frame:end_frame] = True
    voiced_mask &= nonsilent_mask

    voiced_mask = _fill_short_gaps(voiced_mask, MAX_GAP_FRAMES)
    voiced_mask = _drop_short_runs(voiced_mask, MIN_VOICED_FRAMES)

    f0_masked = f0.astype(float, copy=True)
    f0_masked[~voiced_mask] = np.nan
    return f0_masked, f0.astype(float, copy=False), voiced_prob


def summarize_curve(prefix: str, values: np.ndarray) -> dict[str, float]:
    # Summary stats ignoring NaNs
    vec = np.asarray(values, dtype=float).ravel()
    vec = vec[~np.isnan(vec)]
    if vec.size == 0:
        return {
            f"{prefix}_mean": float("nan"),
            f"{prefix}_std": float("nan"),
            f"{prefix}_min": float("nan"),
            f"{prefix}_max": float("nan"),
            f"{prefix}_median": float("nan"),
            f"{prefix}_p10": float("nan"),
            f"{prefix}_p90": float("nan"),
        }
    p10, p90 = np.percentile(vec, [10, 90])
    return {
        f"{prefix}_mean": float(vec.mean()),
        f"{prefix}_std": float(vec.std()),
        f"{prefix}_min": float(vec.min()),
        f"{prefix}_max": float(vec.max()),
        f"{prefix}_median": float(np.median(vec)),
        f"{prefix}_p10": float(p10),
        f"{prefix}_p90": float(p90),
    }


def extract_pitch_features(audio: np.ndarray, sr: int) -> dict[str, float]:
    f0_masked, raw_f0, voiced_prob = compute_pitch(audio, sr)
    voiced_ratio = float(np.sum(~np.isnan(f0_masked))) / float(max(1, f0_masked.size))
    raw_voiced_ratio = float(np.sum(~np.isnan(raw_f0))) / float(max(1, raw_f0.size))

    features = {
        "pitch_frames": float(f0_masked.size),
        "pitch_voiced_ratio": float(voiced_ratio),
        "pitch_raw_voiced_ratio": float(raw_voiced_ratio),
    }
    features.update(summarize_curve("pitch_f0_hz", f0_masked))
    if voiced_prob is not None:
        features["pitch_voicing_prob_mean"] = float(np.nanmean(voiced_prob))
        features["pitch_voicing_prob_min"] = float(np.nanmin(voiced_prob))
        features["pitch_voicing_prob_max"] = float(np.nanmax(voiced_prob))
    return features


In [ ]:
df = pd.read_csv(CSV_PATH)  # metadata for paths + labels
df["emotion"] = df["emotion"].astype(str).str.strip().str.lower()

# Filter: keep xxx, exclude selected classes, and enforce agreement for labeled classes.
# This keeps unlabeled (xxx) examples while dropping sur/fea/oth/dis.
df = df[~df["emotion"].isin(EXCLUDED_EMOTIONS)].copy()
df = df[(df["emotion"] == "xxx") | (df["agreement"] > 0)].copy()
df.shape


(7532, 7)

In [ ]:
CPU_COUNT = os.cpu_count() or 1
COMPUTE_DEVICE = "cpu"  # Force CPU for this CPU-bound extractor
NUM_WORKERS = max(1, CPU_COUNT - 2)
PROGRESS_MIN_INTERVAL = 1.0

print(f"Compute device: {COMPUTE_DEVICE} | extractor_backend=cpu | workers={NUM_WORKERS}")


def process_row(row: dict[str, object]) -> tuple[dict[str, float | str | int] | None, str | None]:
    rel_path = str(row["path"])
    audio_path = AUDIO_ROOT / rel_path
    if not audio_path.exists():
        return None, str(audio_path)

    audio, sr = load_audio(audio_path)
    duration_s = audio.shape[0] / sr
    features = extract_pitch_features(audio, sr)

    record: dict[str, float | str | int] = {
        "path": rel_path,
        "session": int(row["session"]),
        "method": str(row["method"]),
        "gender": str(row["gender"]),
        "emotion": str(row["emotion"]),
        "n_annotators": int(row["n_annotators"]),
        "agreement": int(row["agreement"]),
        "duration_s": float(duration_s),
    }
    record.update(features)
    return record, None


rows: list[dict[str, float | str | int]] = []
missing: list[str] = []
records = df.to_dict(orient="records")

if NUM_WORKERS > 1:
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        mapped = executor.map(process_row, records)
        for record, missing_path in tqdm(mapped, total=len(records), desc="Extracting", unit="file", mininterval=PROGRESS_MIN_INTERVAL):
            if missing_path is not None:
                missing.append(missing_path)
                continue
            if record is not None:
                rows.append(record)
else:
    for record in tqdm(records, total=len(records), desc="Extracting", unit="file", mininterval=PROGRESS_MIN_INTERVAL):
        row_result, missing_path = process_row(record)
        if missing_path is not None:
            missing.append(missing_path)
            continue
        if row_result is not None:
            rows.append(row_result)

feature_df = pd.DataFrame(rows)
feature_df.to_csv(OUT_PATH, index=False)

print(f"Saved: {OUT_PATH}")
print(f"Workers used: {NUM_WORKERS} (cpu_count={CPU_COUNT})")
if missing:
    print(f"Missing audio files: {len(missing)}")
feature_df.shape


Compute device: cpu | extractor_backend=cpu | workers=14


Extracting:   0%|          | 0/7532 [00:00<?, ?file/s]

Saved: f:\Speech-Emotion-Recognition\extracted_features\prosody_pitch\prosody_pitch_features.csv
Workers used: 14 (cpu_count=16)


(7532, 21)